# 02 - Data Cleaning & Star Schema Staging
This notebook outlines the transformation and cleaning logic:
- Deduplication
- Datetime conversion
- Category standardization
- Date reindexing and Forward-Filling of NAVs (handling weekends and holidays)
- Creating dim_investor from transactions
- Ingestion into SQLite


In [1]:
import pandas as pd
import numpy as np
import pathlib
import datetime

base_dir = pathlib.Path('D:/New folder/bluestock_mf_capstone')
raw_dir = base_dir / 'data/raw'
processed_dir = base_dir / 'data/processed'


## Transformation Steps
### A. Categories Standardization
We combine `category` and `sub_category` to create standardized categories.


In [2]:
df_fund = pd.read_csv(raw_dir / '01_fund_master.csv')
df_fund['category'] = df_fund['category'] + ' ' + df_fund['sub_category']
display(df_fund[['amfi_code', 'fund_house', 'scheme_name', 'category']].head())


,amfi_code,fund_house,scheme_name,category
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity Large Cap
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity Large Cap
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity Small Cap
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity Small Cap
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt Gilt


### B. Reindexing & Forward-Filling NAVs (The Crucial Step for Weekends)
Indian mutual funds do not publish NAVs on weekends (Saturdays and Sundays) or market holidays.
We create a continuous daily chronological calendar, reindex, and forward-fill (FFILL) the NAV values.


In [3]:
df_nav_raw = pd.read_csv(raw_dir / '02_nav_history.csv')
df_nav_raw['date'] = pd.to_datetime(df_nav_raw['date'])

# Calendar dates (2022-01-01 to 2026-05-31)
start_date = datetime.date(2022, 1, 1)
end_date = datetime.date(2026, 5, 31)
all_dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Reindex and FFILL for a single fund to see the difference
sample_amfi = 119551
fund_nav_raw = df_nav_raw[df_nav_raw['amfi_code'] == sample_amfi].copy()
print(f'Raw NAV data points for {sample_amfi}: {len(fund_nav_raw)}')

# Set date as index and reindex to full calendar
fund_nav_clean = fund_nav_raw.drop_duplicates(subset=['date']).set_index('date').reindex(all_dates)
print(f'Reindexed data points (with NaNs for weekends): {len(fund_nav_clean)}')

# Forward fill and backward fill
fund_nav_clean['nav'] = fund_nav_clean['nav'].ffill().bfill()
fund_nav_clean['amfi_code'] = sample_amfi
print(f'Clean data points after forward-fill: {len(fund_nav_clean)}')
display(fund_nav_clean.head(10))


Raw NAV data points for 119551: 1150
Reindexed data points (with NaNs for weekends): 1612
Clean data points after forward-fill: 1612


,amfi_code,nav
2022-01-01,119551,54.3856
2022-01-02,119551,54.3856
2022-01-03,119551,54.3856
2022-01-04,119551,54.3474
2022-01-05,119551,54.6869
2022-01-06,119551,55.4550
2022-01-07,119551,55.3692
2022-01-08,119551,55.3692
2022-01-09,119551,55.3692
2022-01-10,119551,55.2835


### C. Numeric Validation
Confirm that essential fields like NAV, AUM, units, and expense ratios are greater than 0.


In [4]:
df_txs = pd.read_csv(raw_dir / '08_investor_transactions.csv')
print('Checking for invalid transactions (Amount <= 0):')
invalid_txs = df_txs[(df_txs['amount_inr'] <= 0)]
print(f'Count of invalid transaction rows: {len(invalid_txs)}')

df_aum = pd.read_csv(raw_dir / '03_aum_by_fund_house.csv')
print('Checking for negative/zero AUM rows:')
invalid_aum = df_aum[df_aum['aum_crore'] <= 0]
print(f'Count of invalid AUM rows: {len(invalid_aum)}')


Checking for invalid transactions (Amount <= 0):
Count of invalid transaction rows: 0
Checking for negative/zero AUM rows:
Count of invalid AUM rows: 0


### D. Dynamic dim_investor Extraction
Extract unique investors from the transaction CSV.


In [5]:
df_investor = df_txs[["investor_id", "gender", "age_group", "annual_income_lakh", "city", "state", "kyc_status"]].drop_duplicates(subset=["investor_id"])
print(f'Unique Investors Extracted: {len(df_investor)}')
display(df_investor.head())


Unique Investors Extracted: 5000


,investor_id,gender,age_group,annual_income_lakh,city,state,kyc_status
0,INV003054,Female,56+,77.1,Hyderabad,Telangana,Verified
1,INV002952,Male,18-25,7.1,Amritsar,Punjab,Verified
2,INV003420,Male,36-45,47.2,Faridabad,Haryana,Verified
3,INV003436,Female,36-45,54.4,Mumbai,Maharashtra,Pending
4,INV004691,Male,26-35,14.5,Noida,Delhi,Pending


## Loading into SQLite database
We trigger the ETL pipeline method.


In [6]:
import sys
sys.path.append('D:/New folder/bluestock_mf_capstone')
from scripts import etl_pipeline, compute_metrics
etl_pipeline.run_etl()
compute_metrics.compute_all_metrics()


Starting Bluestock Mutual Fund ETL Pipeline (Manager Datasets)...

[1/3] Extracting manager raw datasets...
Extraction successful. All 10 manager datasets loaded.

[2/3] Transforming and cleaning datasets...


Transformations and validation checks complete.

[3/3] Loading clean datasets into SQLite Star Schema database...


Cleared existing tables from database for a clean rebuild.
Database schema successfully initialized from schema.sql


2026-06-09 11:43:06,075 - INFO - Initializing Upgraded Performance Metrics Engine...


2026-06-09 11:43:06,251 - INFO - Using Risk-Free Rate (Rf) = 6.50%


...

2026-06-09 11:43:06,912 - INFO - Generated alpha_beta.csv deliverables in reports.


2026-06-09 11:43:06,917 - INFO - Generated worst_drawdown_ranges.csv lookup table.


2026-06-09 11:43:06,928 - INFO - Database fact_performance table successfully reloaded with regression metrics!


2026-06-09 11:43:07,554 - INFO - Precomputed rolling features successfully loaded into fact_features store!


2026-06-09 11:43:07,555 - INFO - Upgraded Performance Metrics Engine completed successfully!
